In [1]:
import sys
sys.path.insert(0, "..")   # raiz do projeto, para importar src/

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')   # Deve vir antes de importar matplotlib.pyplot
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8, 5)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cpu


In [ ]:
dataset = Planetoid(root='/tmp/Planetoid', name='Cora', transform=NormalizeFeatures())
data = dataset[0].to(DEVICE)

print(f"Dataset: {dataset.name}")
print(f"Número de classes:  {dataset.num_classes}")
print(f"Número de features: {dataset.num_node_features}")
print(f"Nós:                {data.num_nodes}")
print(f"Arestas:            {data.num_edges}")
print(f"Grau médio:         {data.num_edges / data.num_nodes:.2f}")
print(f"Nós de treino:      {data.train_mask.sum().item()}")
print(f"Nós de validação:   {data.val_mask.sum().item()}")
print(f"Nós de teste:       {data.test_mask.sum().item()}")

x = data.x.cpu().numpy()
y = data.y.cpu().numpy()
train_mask = data.train_mask.cpu().numpy()
val_mask = data.val_mask.cpu().numpy()
test_mask = data.test_mask.cpu().numpy()

x_train, y_train = x[train_mask], y[train_mask]
x_test, y_test = x[test_mask], y[test_mask]

baseline_results = {}
baselines = {
    'Logistic Regression': LogisticRegression(max_iter=1000, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

for name, model in baselines.items():
    model.fit(x_train, y_train)
    acc = accuracy_score(y_test, model.predict(x_test))
    baseline_results[name] = acc
    print(f"{name:<25} test_acc = {acc:.4f}")

Processing...
Done!


Dataset: Cora
Número de classes:  7
Número de features: 1433
Nós:                2708
Arestas:            10556
Grau médio:         3.90
Nós de treino:      140
Nós de validação:   500
Nós de teste:       1000


In [ ]:
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, JumpingKnowledge
import torch.nn as nn


class GCN(torch.nn.Module):
    """GCN com 3 camadas, BatchNorm, residual e Jumping Knowledge."""
    def __init__(self, in_ch, hidden, out_ch, dropout=0.5, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.input_proj = nn.Linear(in_ch, hidden)
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(num_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.jk = JumpingKnowledge(mode='cat')
        self.classifier = nn.Linear(hidden * num_layers, out_ch)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.input_proj(x))
        outs = []
        for conv, bn in zip(self.convs, self.bns):
            residual = x
            x = F.relu(bn(conv(x, edge_index))) + residual
            x = F.dropout(x, p=self.dropout, training=self.training)
            outs.append(x)
        return self.classifier(self.jk(outs))


class GraphSAGE(torch.nn.Module):
    """GraphSAGE com 3 camadas, BatchNorm, residual e Jumping Knowledge."""
    def __init__(self, in_ch, hidden, out_ch, dropout=0.5, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.input_proj = nn.Linear(in_ch, hidden)
        self.convs = nn.ModuleList([SAGEConv(hidden, hidden) for _ in range(num_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.jk = JumpingKnowledge(mode='cat')
        self.classifier = nn.Linear(hidden * num_layers, out_ch)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.input_proj(x))
        outs = []
        for conv, bn in zip(self.convs, self.bns):
            residual = x
            x = F.relu(bn(conv(x, edge_index))) + residual
            x = F.dropout(x, p=self.dropout, training=self.training)
            outs.append(x)
        return self.classifier(self.jk(outs))


class GAT(torch.nn.Module):
    """GAT com 3 camadas, BatchNorm, residual e Jumping Knowledge."""
    def __init__(self, in_ch, hidden, out_ch, heads=4, dropout=0.5, num_layers=3):
        super().__init__()
        self.dropout = dropout
        self.input_proj = nn.Linear(in_ch, hidden * heads)
        self.convs = nn.ModuleList([
            GATConv(hidden * heads, hidden, heads=heads, concat=True, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden * heads) for _ in range(num_layers)])
        self.jk = JumpingKnowledge(mode='cat')
        self.classifier = nn.Linear(hidden * heads * num_layers, out_ch)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.input_proj(x))
        outs = []
        for conv, bn in zip(self.convs, self.bns):
            residual = x
            x = F.elu(bn(conv(x, edge_index))) + residual
            x = F.dropout(x, p=self.dropout, training=self.training)
            outs.append(x)
        return self.classifier(self.jk(outs))

In [ ]:
def train_gnn(model, data, epochs=300, lr=0.005, weight_decay=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            model.eval()
            with torch.no_grad():
                pred = out.argmax(dim=1)
                val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean()
                best_val_acc = max(best_val_acc, val_acc.item())
    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return test_acc, best_val_acc


NUM_CLASSES = dataset.num_classes
NUM_FEATURES = dataset.num_node_features
HIDDEN = 128
gnn_results = {}

for name, ModelClass, kwargs in [
    ('GCN',       GCN,       {'hidden': HIDDEN, 'dropout': 0.3}),
    ('GraphSAGE', GraphSAGE, {'hidden': HIDDEN, 'dropout': 0.3}),
    ('GAT',       GAT,       {'hidden': 64, 'heads': 4, 'dropout': 0.3}),
]:
    model = ModelClass(NUM_FEATURES, out_ch=NUM_CLASSES, **kwargs).to(DEVICE)
    test_acc, val_acc = train_gnn(model, data)
    gnn_results[name] = test_acc
    print(f"{name:<12} val_acc={val_acc:.4f}  test_acc={test_acc:.4f}")
#+end_src

GCN          val_acc=0.7240  test_acc=0.7370
GraphSAGE    val_acc=0.6780  test_acc=0.6740
GAT          val_acc=0.7240  test_acc=0.7690


In [ ]:
all_results = {**{f"{k} (sem grafo)": v for k, v in baseline_results.items()},
               **{f"{k} (GNN)": v for k, v in gnn_results.items()}}

df = pd.DataFrame(list(all_results.items()), columns=['Modelo', 'Acurácia'])
df = df.sort_values('Acurácia', ascending=False).reset_index(drop=True)

colors = ['#4C72B0' if 'GNN' in m else '#8172B2' for m in df['Modelo']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df['Modelo'], df['Acurácia'], color=colors)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title('GNN vs. Classificadores Tradicionais — Dataset Cora (Planetoid)')
ax.set_ylabel('Acurácia no conjunto de teste')
plt.xticks(rotation=30, ha='right')

for bar, val in zip(bars, df['Acurácia']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f"{val:.1%}", ha='center', va='bottom', fontsize=9)

# Linha de referência: melhor baseline
best_baseline = max(baseline_results.values())
ax.axhline(best_baseline, color='#8172B2', linestyle='--', linewidth=1.2,
           label=f'Melhor baseline: {best_baseline:.1%}')
ax.legend()
fig.tight_layout()
fig.savefig('img/planetoid_comparison.png')
'img/planetoid_comparison.png'
#+end_src

#+begin_src python :results output verbatim drawer
best_gnn = max(gnn_results.values())
best_bl = max(baseline_results.values())
print(f"Melhor GNN:      {best_gnn:.4f} ({max(gnn_results, key=gnn_results.get)})")
print(f"Melhor baseline: {best_bl:.4f} ({max(baseline_results, key=baseline_results.get)})")
print(f"Ganho absoluto:  {best_gnn - best_bl:+.4f} ({(best_gnn - best_bl)*100:+.1f} p.p.)")

Melhor GNN:      0.7690 (GAT)
Melhor baseline: 0.5840 (Random Forest)
Ganho absoluto:  +0.1850 (+18.5 p.p.)
